# Transformer based Adversarial Text Classifier for Prompt Injection Detection


## 0. Data preparation


In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")  

X_full = pd.read_parquet(DATA_DIR / "train.parquet")
X_test = pd.read_parquet(DATA_DIR / "test.parquet")
split_idx = int(0.85 * len(X_full))

X_train = X_full.iloc[:split_idx]
X_val = X_full.iloc[split_idx:]

In [10]:
total_length = len(X_full) + len(X_test)
print("fraction of Train data: ", len(X_train)/total_length)
print("fraction of Val data: ", len(X_val)/total_length)
print("fraction of Test data: ", len(X_test)/total_length)

fraction of Train data:  0.7009063444108762
fraction of Val data:  0.12386706948640483
fraction of Test data:  0.17522658610271905


## 1. The building blocks of the Transformer

0. Tokenizer
1. Multi-Head Self-Attention
2. MLP
3. Block

Implementations: `self_attention.py` (multi-head attention), `transformer.py` (`PositionwiseFFN`, `Block`, `BinaryClassifier`, `Config`).

`BinaryClassifier.forward` returns logits of shape **`(batch_size, num_labels)`** (typically `(B, 2)`). Use **`nn.CrossEntropyLoss(logits, labels)`** with labels of shape `(B,)`.

For padded batches, either pass **`attention_mask`** with `1` on real tokens and `0` on padding, or set **`Config.pad_token_id`** / pass **`pad_token_id=...`** so mean pooling ignores padded positions.

In [ ]:
import torch
from torch import nn

from transformer import BinaryClassifier, Config

# Toy example: padded token matrix; align pad_id with tokenizer (reserve a padding id in vocab if needed)
pad_id = 0
config = Config(
    vocab_size=50,
    vector_dim=32,
    block_size=16,
    number_of_transformer_blocks=2,
    number_of_attention_heads=2,
    pad_token_id=pad_id,
)
model = BinaryClassifier(config)
model.eval()

x = torch.tensor(
    [[3, 4, 5, 0, 0], [7, 8, 9, 10, 0]],
    dtype=torch.long,
)
attention_mask = (x != pad_id).float()

logits_from_mask = model(x, attention_mask=attention_mask)
logits_from_pad_kw = model(x, pad_token_id=pad_id)

assert logits_from_mask.shape == (2, config.num_labels)
assert torch.allclose(logits_from_mask, logits_from_pad_kw)

criterion = nn.CrossEntropyLoss()
labels = torch.tensor([0, 1], dtype=torch.long)
loss = criterion(logits_from_mask, labels)
loss.backward()

print("logits shape:", tuple(logits_from_mask.shape), "| loss:", float(loss.detach()))
